# Caustic module usage

This notebook shows the real workflow behind the caustic helper module:

1. acquire a raw beam image from Shadow,
2. turn that image ticket into a `Histogram2DAnalyzer`,
3. compute moment-based estimates,
4. run the Gaussian fit explicitly,
5. use the fitted widths in a caustic scan.

The key point is: the object returned by `save_image()` is not itself a fitted analyzer. It is a data container. The fit lives in `Histogram2DAnalyzer`.

In [ ]:
import os
import sys

sys.path.append(os.path.expanduser("~/repos/cax-control"))

from caustic import ana_from_image, fit_beam_image, image_caustic, fit_caustic_hyperbola
from shadowpy.si_beamlines import CAXSim
from shadowpy.utils import save_image, image_from_beam
from image_statistics import Histogram2DAnalyzer
import numpy as np
import matplotlib.pyplot as plt
# Example only: replace these with your real beam and element objects.
# beam = cax.dvf_B1.beam.duplicate()
# frame = cax.dvf_B1.frame

In [ ]:
cax = CAXSim(image_size=(100, 100))

In [ ]:
# Step 1: produce a raw image from a beam and a frame.
image = cax.dvf_B1.image
# image is a container around the ticket, not a fitted analyzer.
# print(type(image), image.histogram.shape)

# Example placeholder assignment to show the expected workflow.
# image = None

In [ ]:
# Step 2: turn the raw image into a Histogram2DAnalyzer.
# This is the step that creates the hprm_momenta / hprm_fitting data.
ana = ana_from_image(image)
ana.print_stats(ana.hprm_project)
# Example placeholder.
# ana = None

In [ ]:
ana.plot(hprm=ana.hprm_project)

In [ ]:
# Step 3: compute moment estimates and then run the fit explicitly.
# This is the important part: the beam is not fitted automatically when the raw
# image is saved. The Gaussian fit must be run in the analyzer.
moments = ana.compute_momenta()
fit = ana.fit(hprm=moments)
ana.print_stats(fit)
# Example placeholder.
# fit = None

In [ ]:
ana.plot(hprm=fit)

In [ ]:
# Convenience helper that does the exact same fit pipeline in one call.
# fit_beam_image(image)
# This returns the analyzer object with hprm_fitting populated.

# Example placeholder.
# ana_fit = fit_beam_image(image)
# print(ana_fit.hprm_fitting['fwhmx'])

In [ ]:
cax = CAXSim(image_size=(100, 100))

In [ ]:
cax.mirror.ry = 25.255e-3#0.0#0.1#25.25e-3#16.82e-3
cax.mirror.tx = 0.0#-2.0#-1 #-0.135

In [ ]:
cax.trace()

In [ ]:
# Step 4: caustic scan using the fitted widths.
# The generic caustic helper does not operate on an already-fitted image
# object. It evaluates a beam at many distances and fits a hyperbola to the
# measured widths and their uncertainties.

distances = np.linspace(-450, 150, 20)
result = image_caustic(cax.dvf_B1.beam, cax.dvf_B1.frame, distances, axes=("x", "y"), max_workers=10)

print(result.focus_x, result.focus_y)
print(result.astigmatism)

# 

# Example placeholder.
# result = None

In [ ]:
# %timeit image_caustic(cax.dvf_B1.beam, cax.dvf_B1.frame, distances, axes=("x", "y"), max_workers=1)

In [ ]:
result.x.fit_params

In [ ]:
# focus fit uncertainty
np.sqrt(result.x.covariance[1, 1]), np.sqrt(result.y.covariance[1, 1])


In [ ]:
np.sqrt(result.x.covariance[1, 1])*100/600, np.sqrt(result.y.covariance[1, 1])*100/600


In [ ]:
result.x.covariance

In [ ]:
fig, _, _ = result.plot()
original_title = fig.axes[0].get_title()
fig.axes[0].set_title(original_title + " mm" + f" (RY = {cax.mirror.ry*1000:.3f} μrad — TX = {cax.mirror.tx:.3f} mm)")

In [ ]:
ana.hprm_fitting['errors']['sigx'], ana.hprm_fitting['errors']['sigy']

In [ ]:
result.x.uncertainties, result.y.uncertainties

In [ ]:
ana = fit_beam_image(cax.dvf_B1.image)
ana.plot(hprm=ana.hprm_fitting)

In [ ]:
fig.savefig("/home/gabriel.amici/Documents/CEC/caustic-ry-25.pdf", dpi=300, bbox_inches="tight")
fig.savefig("/home/gabriel.amici/Documents/CEC/caustic-ry-25.png", dpi=300, bbox_inches="tight")

In [ ]:
beamr = cax.dvf_B1.beam.duplicate()

In [ ]:
%timeit beamr=cax.dvf_B1.beam.duplicate() ; beamr.retrace(np.random.randint(-100, 100)) ; image = image_from_beam(beamr, cax.dvf_B1.frame, image_size=(100, 100)) ; ana = fit_beam_image(image)

## Practical rule

- `save_image()` and `Image` only give the raw histogram ticket.
- `Histogram2DAnalyzer` is where the beam fitting happens.
- `caustic.py` is only the scan-and-fit helper around those measurements.
- The fitted caustic result is used to estimate the focus minimum; the image-level fit still drives the width and uncertainty values.